# OLD Scraper logic

In [91]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.indiabix.com/current-affairs"
PATH = "questions-and-answers/"

def get_soup(BASE_URL, PATH):
    url = f"{BASE_URL}/{PATH}"
    print(url)
    res = requests.get(url)
    if res.status_code == 200:
        soup = BeautifulSoup(res.text, "html.parser")
    else:
        print(res.status_code)
    
    return soup

soup = get_soup(BASE_URL, PATH)

https://www.indiabix.com/current-affairs/questions-and-answers/


In [103]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from threading import Lock
from bs4 import BeautifulSoup
import sqlite3
import json
import re
import time
import random

# ============================================
# CONFIG
# ============================================
BASE_URL = "https://www.indiabix.com/current-affairs"
PATH = "2026-05-23"

START_DATE = "2025-11-01"
END_DATE   = "2026-05-23"

MAX_WORKERS = 5
MAX_RETRIES = 5

SQLITE_DB = "scraper.db"
OUTPUT_JSON = "questions_by_date.json"

db_lock = Lock()

# ============================================
# DB
# ============================================

conn = sqlite3.connect(SQLITE_DB, check_same_thread=False)

conn.execute("""
CREATE TABLE IF NOT EXISTS scrape_status(
    date TEXT PRIMARY KEY,
    status TEXT,
    question_count INTEGER,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()


def already_scraped(date):
    row = conn.execute(
        """
        SELECT status
        FROM scrape_status
        WHERE date=?
        """,
        (date,)
    ).fetchone()

    return row and row[0] == "success"


def mark_success(date, count):
    with db_lock:
        conn.execute(
            """
            INSERT OR REPLACE INTO scrape_status
            (date,status,question_count)
            VALUES(?,?,?)
            """,
            (date, "success", count)
        )
        conn.commit()


def mark_failed(date):
    with db_lock:
        conn.execute(
            """
            INSERT OR REPLACE INTO scrape_status
            (date,status,question_count)
            VALUES(?,?,?)
            """,
            (date, "failed", 0)
        )
        conn.commit()


# ============================================
# HELPERS
# ============================================
def get_soup(BASE_URL, PATH):
    url = f"{BASE_URL}/{PATH}"
    print(url)
    res = requests.get(url)
    if res.status_code == 200:
        soup = BeautifulSoup(res.text, "html.parser")
    else:
        print(res.status_code)
    
    return soup

def clean_text(text):

    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def generate_dates(start, end):

    start = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")

    current = start

    while current <= end:
        yield current.strftime("%Y-%m-%d")
        current += timedelta(days=1)


# ============================================
# QUESTION PARSER
# ============================================

def parse_questions(soup):

    all_questions = []

    question_divs = soup.find_all(
        "div",
        class_="bix-div-container"
    )

    for q in question_divs:

        item = {
            "question_no": "",
            "question": "",
            "options": {},
            "correct_answer": "",
            "correct_answer_text": "",
            "explanation": "",
            "category": ""
        }

        # Question Number
        div = q.find("div", class_="bix-td-qno")
        if div:
            item["question_no"] = clean_text(div.get_text())

        # Question
        div = q.find("div", class_="bix-td-qtxt")
        if div:
            item["question"] = clean_text(div.get_text())

        # Options
        div = q.find("div", class_="bix-tbl-options")

        if div:

            options = []

            for txt in div.stripped_strings:

                txt = clean_text(txt)

                if txt and txt not in options:
                    options.append(txt)

            options = options[:4]

            labels = ["A", "B", "C", "D"]

            for label, option in zip(labels, options):
                item["options"][label] = option

        # Correct answer
        div = q.find("div", class_="bix-td-miscell")

        if div:

            inp = div.find("input")

            if inp:

                answer = inp.get("value", "").strip()

                item["correct_answer"] = answer

                item["correct_answer_text"] = (
                    item["options"].get(answer, "")
                )

        # Explanation
        div = q.find(
            "div",
            class_="bix-ans-description"
        )

        if div:
            item["explanation"] = clean_text(
                div.get_text()
            )

        # Category
        div = q.find("div", class_="explain-link")

        if div:

            category = clean_text(
                div.get_text()
            )

            if ":" in category:
                category = category.split(
                    ":",
                    1
                )[1].strip()

            item["category"] = category

        all_questions.append(item)

    return all_questions


# ============================================
# FETCH SINGLE DATE
# ============================================

def fetch_date(date):

    if already_scraped(date):
        print(f"SKIP {date}")
        return None

    for attempt in range(MAX_RETRIES):

        try:

            print(
                f"FETCH {date} "
                f"(attempt {attempt+1})"
            )

            soup = get_soup(BASE_URL, date)

            questions = parse_questions(soup)

            mark_success(
                date,
                len(questions)
            )

            time.sleep(
                random.uniform(
                    0.2,
                    0.8
                )
            )

            return {
                "date": date,
                "questions": questions
            }

        except Exception as e:

            print(
                f"ERROR {date}: {e}"
            )

            wait = 2 ** attempt

            time.sleep(wait)

    mark_failed(date)

    return None


# ============================================
# LOAD EXISTING OUTPUT
# ============================================

try:

    with open(
        OUTPUT_JSON,
        "r",
        encoding="utf-8"
    ) as f:

        final_data = json.load(f)

except:

    final_data = {}


# ============================================
# PARALLEL SCRAPE
# ============================================

dates = list(
    generate_dates(
        START_DATE,
        END_DATE
    )
)

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {
        executor.submit(
            fetch_date,
            date
        ): date
        for date in dates
    }

    for future in as_completed(futures):

        result = future.result()

        if result is None:
            continue

        date = result["date"]

        final_data[date] = result["questions"]

        # incremental save
        with open(
            OUTPUT_JSON,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                final_data,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"Saved {date} "
            f"({len(result['questions'])} questions)"
        )

print("DONE")

SKIP 2025-11-01
SKIP 2025-11-02
SKIP 2025-11-03
SKIP 2025-11-04
FETCH 2025-11-07 (attempt 1)
https://www.indiabix.com/current-affairs/2025-11-07
SKIP 2025-11-06
SKIP 2025-11-05
FETCH 2025-11-10 (attempt 1)
https://www.indiabix.com/current-affairs/2025-11-10
SKIP 2025-11-11
SKIP 2025-11-09
FETCH 2025-11-14 (attempt 1)
https://www.indiabix.com/current-affairs/2025-11-14
SKIP 2025-11-13
SKIP 2025-11-12
SKIP 2025-11-15
SKIP 2025-11-16
FETCH 2025-11-18 (attempt 1)
https://www.indiabix.com/current-affairs/2025-11-18
SKIP 2025-11-17
SKIP 2025-11-19
SKIP 2025-11-20
SKIP 2025-11-21
SKIP 2025-11-22
SKIP 2025-11-23
SKIP 2025-11-24
SKIP 2025-11-25
SKIP 2025-11-26
SKIP 2025-11-27
SKIP 2025-11-28
SKIP 2025-11-29
SKIP 2025-11-30
SKIP 2025-12-01
SKIP 2025-12-02
SKIP 2025-12-03
SKIP 2025-12-04
SKIP 2025-12-05
SKIP 2025-12-06
SKIP 2025-12-07
SKIP 2025-12-08
SKIP 2025-12-09
SKIP 2025-12-10
SKIP 2025-12-11
SKIP 2025-12-12
SKIP 2025-12-13
SKIP 2025-12-14
SKIP 2025-12-15
SKIP 2025-12-16
SKIP 2025-12-17
SKIP

InterfaceError: bad parameter or other API misuse

# 2nd Version

RuntimeError: asyncio.run() cannot be called from a running event loop